In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import joblib

In [3]:
df=pd.read_csv('../datasets/olist_master_dataset.csv')
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'total_payment_value', 'payment_installments', 'payments_method_count',
       'primary_payment_type', 'review_score', 'review_comment_title',
       'review_comment_message', 'product_category', 'is_delivered'],
      dtype='object')

In [4]:
df['customer_unique_id'].nunique()

95420

In [5]:
df['product_id'].nunique()

32951

In [6]:
df1=df[['customer_unique_id','product_id']]

In [7]:
df1.head()

,customer_unique_id,product_id
0,7c396fd4830fd04220f754e42b4e5bff,87285b34884572647811a353c7ac498a
1,af07308b275d755c9edb36a90c618231,595fac2a385ac33a80bd5114aec74eb8
2,3a653a41f6f9fc3d2a113cf8398680e8,aa4383b373c6aca5d8797843e5594415
3,7c142cf63193a1473d2e66489a9ae977,d0b61bfb1de832b15ba9d266ca96e5b0
4,72632f0f9dd73dfee390c9b22eb56dd6,65266b2da20d04dbe00c5c2d3bb7859e


In [8]:
df1.shape

(113314, 2)

In [9]:
df1.isna().sum()

customer_unique_id    0
product_id            0
dtype: int64

In [10]:
df_unique = df1.drop_duplicates(subset=['customer_unique_id', 'product_id']).reset_index(drop=True)

In [11]:
unq_users=df_unique['customer_unique_id'].unique()

In [12]:
unq_products=df_unique['product_id'].unique()

In [13]:
user_to_index={uid:idx for idx,uid in enumerate(unq_users)}

In [14]:
product_to_index={pid:idx for idx,pid in enumerate(unq_products)}

In [15]:
index_to_product = {idx: pid for pid, idx in product_to_index.items()}

In [16]:
row_indices=df_unique['customer_unique_id'].map(user_to_index).to_numpy(dtype=np.int32)

In [17]:
col_indices=df_unique['product_id'].map(product_to_index).to_numpy(dtype=np.int32)

In [18]:
data=np.ones(len(row_indices),dtype=np.int8)

In [19]:
user_item_csr=csr_matrix((data,(row_indices,col_indices)),shape=(len(user_to_index),len(product_to_index)),dtype=np.int8)

In [20]:
n_users = len(unq_users)
n_products = len(unq_products)

In [21]:
print("unique users:", n_users)
print("unique products:", n_products)
print("user_item_csr.shape:", user_item_csr.shape)
print("non-zero interactions (nnz):", user_item_csr.nnz)
print("sparsity (fraction non-zero):", user_item_csr.nnz / (n_users * n_products))

unique users: 95420
unique products: 32951
user_item_csr.shape: (95420, 32951)
non-zero interactions (nnz): 101987
sparsity (fraction non-zero): 3.243671056674214e-05


In [22]:
item_user_matrix = user_item_csr.T

In [23]:
item_user_matrix.shape

(32951, 95420)

In [24]:
from sklearn.neighbors import NearestNeighbors

In [25]:
knn=NearestNeighbors(metric='cosine',algorithm='brute',n_jobs=-1)

In [26]:
knn.fit(item_user_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1)

In [27]:
dist,indices=knn.kneighbors(item_user_matrix[255],n_neighbors=6)
indices=indices.flatten()

In [28]:
neighbors=[index_to_product[i] for i in indices]

In [29]:
neighbors

['9ddd762ee8a13576a809dc66f22aa2b5',
 'ecc1323a42fb16f485902b43aef03a03',
 'dd2a86c701f0442fa851c5391a5b81fe',
 '41da75141264c3bde21ecea85a4cb8b7',
 '79351053555d5b98a09738e68f891f1a',
 '429e7401fafb76436f15e86498bd7364']

In [30]:
from joblib import dump

In [31]:
dump(knn,'../backend/models/recommendation_model.joblib')

['../backend/models/recommendation_model.joblib']

In [32]:
from scipy.sparse import save_npz

In [33]:
save_npz('../backend/models/item_user_matrix.npz',item_user_matrix)

In [34]:
avg_review=df.groupby('product_id')['review_score'].mean().reset_index(name='avg_review_score')

In [35]:
avg_review['avg_review_score'] = avg_review['avg_review_score'].apply(lambda x: round(x, 1))

In [37]:
avg_review.to_csv('../datasets/avg_product_review_score.csv')